<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SplatTransform_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## SplatTransform — 3DGS post-processor (PlayCanvas, MIT)

[PlayCanvas `splat-transform`](https://github.com/playcanvas/splat-transform) is the missing piece for shipping 3D Gaussian Splat assets to a game engine. Takes a raw 3DGS `.ply` (like the 150-250 MB output from [TripoSplat_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/TripoSplat_Colab.ipynb)) and converts it to game-engine-friendly formats:

- **`.sog`** (PlayCanvas native) — ~10× compression via SH coefficient bundling into WebP textures. Best for [PlayCanvas Engine](https://playcanvas.com/) and any GPU-accelerated viewer.
- **`.spz`** (Niantic) — smallest cross-platform splat format. Best for mobile AR / Scaniverse.
- **`.glb` + `KHR_gaussian_splatting`** (glTF 2.0 extension) — the **new Khronos standard** for shipping splats. Works in Three.js (gsplat.js), Babylon.js, and PlayCanvas today; Unity/Unreal glTF importer support is coming.
- **`.ply`** (original / decimated / SH-stripped) — lossless or reduced. Universal fallback.
- **`.html`** self-contained viewer — drop into any static host for instant web preview.

**Not a mesh generator.** The only mesh `splat-transform` produces is a **voxel collision mesh** (`.collision.glb` from marching-cubes, for runtime physics — not a textured visual mesh). For visual mesh extraction from 3DGS, see [SuGaR_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SuGaR_Colab.ipynb) and [GauStudio_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/GauStudio_Colab.ipynb). For game-ready textured meshes from a single image, use [Pixal3D_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Pixal3D_Colab.ipynb) instead.

**What it does:**

- **Step 1** — install Node 22+ and `splat-transform` from npm (PlayCanvas MIT, no Python wrapper exists).
- **Step 2** — Drive cache prologue + GPU check (WebGPU optional; most steps have a CPU fallback).
- **Step 3** — **batch convert** a folder of TripoSplat PLYs into all 4 formats (SOG, SPZ, GLB, PLY) with size + compression-ratio reports.
- **Step 4** — **decimate** (reduce Gaussian count for web previews / low-LOD) and **strip SH bands** (drop color detail for faster loads).
- **Step 5** — build **LOD chains** (streamed SOG for progressive loading in PlayCanvas viewer).
- **Step 6** — generate **voxel collision meshes** (`.collision.glb` for runtime physics) + GPU rasterized **turntable previews** (`.webp`).
- **Step 7** — Drive mirror of the whole export folder.
- **Step 8** — keep-alive.
- **Step 9** — help / format reference / engine compatibility / known issues.

**Compute:** Node 22+ runtime (Colab needs `apt-get install nodejs`). GPU optional — most steps work on CPU; SOG SH k-means benefits from GPU (5–10× faster). L4 / T4 both fine. First run: ~2-3 min install. Subsequent: instant per file.

**License:** MIT (PlayCanvas Ltd.). Commercial-OK, no attribution-in-UI requirement, no copyleft.

In [ ]:
# @title STEP 1 — Install Node 22+ and PlayCanvas splat-transform
# @markdown Installs the latest Node.js (the `splat-transform` CLI requires
# @markdown Node 18+ but 22+ is recommended for SOG SH k-means GPU paths), then
# @markdown `splat-transform` from npm. ~2-3 min on first run.

import os, sys, subprocess, time, json

print('[splat-transform] MIT license, PlayCanvas Ltd. Commercial-OK.')
print('[splat-transform] https://github.com/playcanvas/splat-transform')
print(flush=True)

# ── Install Node 22+ via NodeSource ────────────────────────────────────────────
# Colab's default nodejs may be 18/20; NodeSource gives us 22 LTS.
def _node_version():
    try:
        return subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return ''

nv = _node_version()
if not nv.startswith('v22') and not nv.startswith('v23') and not nv.startswith('v24'):
    print(f'[install] Current node = {nv!r}. Installing Node 22 LTS via NodeSource...')
    subprocess.run(['apt-get', 'remove', '-y', '-qq', 'nodejs', 'nodejs-legacy'], check=False)
    subprocess.run('curl -fsSL https://deb.nodesource.com/setup_22.x | bash -',
                   shell=True, check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'nodejs'], check=True)
    nv = _node_version()
print(f'[install] Node = {nv!r}')

# ── Install splat-transform globally ────────────────────────────────────────────
# npm i -g installs the CLI into PATH so `splat-transform` is callable
# directly from `subprocess` (and from `!` shell calls in other cells).
t0 = time.time()
print('[npm] Installing @playcanvas/splat-transform ...', flush=True)
subprocess.run(['npm', 'install', '-g', '@playcanvas/splat-transform'], check=True)
print(f'[npm] Install: {time.time()-t0:.1f}s', flush=True)

# ── Sanity check ───────────────────────────────────────────────────────
import shutil
cli_path = shutil.which('splat-transform')
print(f'[verify] splat-transform: {cli_path!r}')
if not cli_path:
    print('[verify] WARNING: splat-transform not on PATH. The CLI should be at /usr/local/bin/splat-transform.')
    print('[verify] If subsequent steps fail with "command not found", try running `npx splat-transform ...` instead.')

# ── Version + a no-op conversion to confirm the install works ─────────────────────
result = subprocess.run(['splat-transform', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'[verify] Version: {result.stdout.strip()}')
else:
    print(f'[verify] Version check failed: {result.stderr.strip()}')
    print('[verify] (May still work for actual conversions. Try STEP 3 anyway.)')

# Optional: install texture-tool sister project too, for completeness
subprocess.run(['npm', 'install', '-g', '@playcanvas/texture-tool'], check=False)
print('[npm] Installed texture-tool (sister project for 2D textures).')

print(f'\n[Done] STEP 1 complete in {time.time()-t0:.0f}s. Move on to STEP 2 (Drive cache).')


### Engine Preset — when to use it

If you're feeding this notebook's output into a **WebGPU Gaussian-splat game engine** that:

- Loads bundled `.sog` (SOG v2) directly
- Auto-orients / normalizes object-scale assets at load time (180° upright flip, recenter, fit-to-size, ground at y=0)
- Shader evaluates only DC + degree-1 SH

then enable **STEP 2b → ENGINE_PRESET = True** and pick `ASSET_CLASS = 'prop'` or `'environment'`.

#### Prop vs environment — the critical distinction

| Asset class | What it is | Engine loads it as | Pre-grounding? | Streaming octree? |
|---|---|---|---|---|
| **prop** | Object-scale asset (hero, weapon, furniture) | Auto-oriented at load (180° flip + recenter + fit-to-size + y=0 ground) | **NO** — pre-grounding double-applies and breaks positioning | No — just bundled SOG + distance-swapped LOD |
| **environment** | Big world capture (terrain, city block, room) | Loaded in world coords as-is | Optional but recommended for collision-mesh stability | **YES** — streamed-SOG octree (`lod-meta.json` + per-chunk SOGs) |

**The number one mistake to avoid:** pre-grounding a prop for an auto-orienting engine. The engine will apply its own 180° flip + recenter on top of your transform, and the prop will end up buried underground or floating in the air.

#### What the preset does (and doesn't do)

When `ENGINE_PRESET = True`:

- Biais `WRITE_SOG=True`, `WRITE_SPZ=False`, `WRITE_GLB=False`, `WRITE_PLY=False` (still individually toggleable in STEP 3)
- Bias `STRIP_SH_BANDS=3` (DC-only; the shader's max). Use `2` to keep degree-1 SH for hero assets.
- Bias `RE_GROUND_ON_IMPORT` based on `ASSET_CLASS` (False for props, True for environments)
- Prints a summary so you can see exactly which defaults were applied

When `ENGINE_PRESET = False` (default): every existing param behaves exactly as before. No regressions for existing users.

For environment captures, also run **STEP 5b (Streamed-SOG Octree)** after STEP 3 — it builds the engine's native `lod-meta.json` from decimated SOGs.


In [ ]:
# @title STEP 2b — Engine Preset (optional, default OFF)
# @markdown Single-toggle preset that biases every output choice toward the
# @markdown needs of a WebGPU/PlayCanvas Gaussian-splat game engine that ingests
# @markdown bundled `.sog` (SOG v2), auto-orients/normalizes object-scale assets at
# @markdown load time, and whose shader evaluates only DC + degree-1 SH.
# @markdown
# @markdown **When OFF (default):** every param below has no effect. STEP 3 uses
# @markdown its own defaults (SOG+SPZ+GLB+PLY all written, no SH strip by default).
# @markdown This is the safe behavior for existing users.
# @markdown
# @markdown **When ON:** STEP 3 reads the recommended values from this cell and
# @markdown biases its output formats + SH strip accordingly. You can still flip
# @markdown any individual param afterwards in STEP 3 — the preset only seeds the
# @markdown recommended defaults.
# @markdown
# @markdown See the markdown cell above for the full prop-vs-environment distinction
# @markdown (the most important caveat: don't pre-ground props for auto-orienting engines).
ENGINE_PRESET = False  # @param {type:"boolean"}
# @markdown *Master toggle. Default OFF so existing users see no behavior change.*
ASSET_CLASS = 'prop'  # @param ["prop", "environment"]
# @markdown *Informs the preset which defaults to use. 'prop' = object-scale asset
# @markdown (auto-oriented by the engine at load). 'environment' = big world capture
# @markdown (loaded in world coords; the streamed-SOG octree + grounding make sense).*
PRESET_SH_STRIP = 3  # @param {type:"slider", min:0, max:3, step:1}
# @markdown *SH band cap when preset is ON. 3 = DC-only (recommended for most assets
# @markdown in the engine; the shader evaluates DC + degree-1 by default).
# @markdown Use 2 to KEEP degree-1 SH for hero assets; use 0 for any SH (max quality, biggest files).*
PRESET_WRITE_SOG = True  # @param {type:"boolean"}
# @markdown *When preset ON, force-enable SOG output (engine's preferred format).*
PRESET_WRITE_SPZ = False  # @param {type:"boolean"}
# @markdown *When preset ON, skip SPZ by default (engine doesn't ingest it).*
PRESET_WRITE_GLB = False  # @param {type:"boolean"}
# @markdown *When preset ON, skip GLB+KHR_gaussian_splatting by default (still available
# @markdown in STEP 3 if you want it for Three.js / Babylon / glTF tools).*
PRESET_WRITE_PLY = False  # @param {type:"boolean"}
# @markdown *When preset ON, skip PLY passthrough by default (still available in STEP 3).*
PRESET_GROUNDING = False  # @param {type:"boolean"}
# @markdown *Pre-grounding/recenter/PCA-align for STEP 6 colliders. Default False for
# @markdown object props (the engine auto-orients). Set True for environment captures
# @markdown (they load in world coords).*
# Set globals that STEP 3 will read.
# When preset OFF: globals stay None and STEP 3 uses its existing defaults unchanged.
# When preset ON: globals set the recommended values; STEP 3 still lets the user flip
# individual toggles in its own cell (which override these).
if ENGINE_PRESET:
    _preset_write_sog = PRESET_WRITE_SOG
    _preset_write_spz = PRESET_WRITE_SPZ
    _preset_write_glb = PRESET_WRITE_GLB
    _preset_write_ply = PRESET_WRITE_PLY
    _preset_sh_bands = PRESET_SH_STRIP
    _preset_grounding = PRESET_GROUNDING
    _summary = (
        f'[preset] Engine preset ON, assetClass={ASSET_CLASS}.\n'
        f'[preset]   WRITE_SOG={_preset_write_sog}, WRITE_SPZ={_preset_write_spz},\n'
        f'[preset]   WRITE_GLB={_preset_write_glb}, WRITE_PLY={_preset_write_ply}.\n'
        f'[preset]   STRIP_SH_BANDS={_preset_sh_bands}.\n'
        f'[preset]   RE_GROUND_ON_IMPORT={_preset_grounding}.\n'
        f'[preset]   (STEP 3’s individual toggles can still override these.)\n'
    )
    if ASSET_CLASS == 'prop':
        _summary += '[preset]   For object props: the engine auto-orients them at load.\n[preset]     Pre-grounding would double-apply and break positioning.\n'
    else:
        _summary += '[preset]   For environment captures: streaming + grounding is right.\n[preset]     Use STEP 5b (Streamed-SOG Octree) for these.\n'
    print(_summary)
else:
    _preset_write_sog = None
    _preset_write_spz = None
    _preset_write_glb = None
    _preset_write_ply = None
    _preset_sh_bands = None
    _preset_grounding = None
    print('[preset] Engine preset OFF (default). STEP 3 uses its own defaults. No change.')


In [ ]:
# @title STEP 3 — Batch Convert: TripoSplat PLY → SOG / SPZ / GLB / PLY
# @markdown Scans a folder of TripoSplat PLYs and converts each into 4 game-friendly
# @markdown formats. Reports original size, each output size, and compression ratio.

import os, sys, time, json, shutil, subprocess, pathlib
from tqdm import tqdm
from IPython.display import FileLink, display

INPUT_DIR = '/content/drive/MyDrive/AEI_3D_Out/TripoSplat'  # @param {type:"string"}
# @markdown *Folder containing TripoSplat .ply files. Recursive scan.*
OUTPUT_DIR = '/content/splat_out'  # @param {type:"string"}
# @markdown *Where to write converted files. Each input .ply becomes up to 4 outputs.*
RECURSIVE = True  # @param {type:"boolean"}
# @markdown *Scan subfolders for input PLYs.*
MAX_ITEMS = 0  # @param {type:"integer"}
# @markdown *Cap the number of PLYs to process. 0 = no cap.*
DRIVE_MIRROR = True  # @param {type:"boolean"}
# @markdown *Mirror OUTPUT_DIR to Drive at the end.*
# @markdown **Output formats** (toggle each one):
WRITE_SOG = True  # @param {type:"boolean"}
# @markdown *`.sog` (PlayCanvas native, GPU-accelerated SH k-means, ~10x compression).*
WRITE_SPZ = True  # @param {type:"boolean"}
# @markdown *`.spz` (Niantic, smallest cross-platform, mobile-friendly).*
WRITE_GLB = True  # @param {type:"boolean"}
# @markdown *`.glb` with `KHR_gaussian_splatting` extension (glTF 2.0 standard, Three.js + PlayCanvas).*
WRITE_PLY = True  # @param {type:"boolean"}
# @markdown *`.ply` lossless passthrough (or decimated if you set -F below).*
DECIMATE_TO = 0  # @param {type:"integer"}
# @markdown *0 = keep all Gaussians. 100000 = decimate to 100k for low-LOD. Applied to SOG/SPZ/GLB outputs.*
STRIP_SH_BANDS = 0  # @param {type:"slider", min:0, max:3, step:1}
# @markdown *0 = keep all SH bands. 1-3 = drop higher bands for faster loads (less color detail). 3 = DC only.*
GPU_MODE = True  # @param {type:"boolean"}
# @markdown *Use GPU for SOG SH k-means (5-10x faster). Set False for CPU fallback.*

# @markdown *Use GPU for SOG SH k-means (5-10x faster). Set False for CPU fallback.*
GPU_MODE = True  # @param {type:"boolean"}

# --- Batch hardening (additive, all default-safe) ---
PARALLEL_WORKERS = 2  # @param {type:"slider", min:1, max:8, step:1}
# @markdown *Number of concurrent splat-transform workers. 1 = strictly serial (safest,
# @markdown easiest to read). 2-4 = typical sweet spot on Colab (saves wall time on
# @markdown thousands of files). Higher = more RAM + GPU contention; rarely helpful.*
RESUME = True  # @param {type:"boolean"}
# @markdown *Skip a source PLY if all its enabled-format outputs already exist with
# @markdown non-zero size. Pair with FORCE_REDO to regenerate. Survives Colab disconnects.*
FORCE_REDO = False  # @param {type:"boolean"}
# @markdown *Force regeneration even if outputs exist. Use after changing decimation/SH params.*
VALIDATE_OUTPUTS = True  # @param {type:"boolean"}
# @markdown *Re-open each produced .sog and confirm it parses + splat count > 0.
# @markdown Catches silent corruption early. ~2% overhead per file.*
EMIT_MANIFEST = True  # @param {type:"boolean"}
# @markdown *Write `<output_dir>/manifest.json` with per-source metadata (paths, sizes,
# @markdown compression ratios, splat counts, bounding box, assetClass). Lets the
# @markdown engine's upload step automate asset records + LOD variants.*
ASSET_CLASS_THRESHOLD = 1000000  # @param {type:"integer"}
# @markdown *Source splat-count threshold above which an asset is auto-classified as
# @markdown 'environment' instead of 'prop'. Default 1M = typical cutoff for object-scale
# @markdown vs big captures. Lower = more things classified as environment.*

input_dir = pathlib.Path(INPUT_DIR)
output_dir = pathlib.Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

if not input_dir.exists():
    raise SystemExit(f'[convert] {input_dir} not found. Run TripoSplat_Colab.ipynb Step 7 first to generate PLYs.')

# Find PLYs
if RECURSIVE:
    plys = sorted(input_dir.rglob('*.ply'))
else:
    plys = sorted(input_dir.glob('*.ply'))
if not plys:
    raise SystemExit(f'[convert] No .ply files in {input_dir} (recursive={RECURSIVE}). Run TripoSplat first.')
if MAX_ITEMS:
    plys = plys[:MAX_ITEMS]
print(f'[convert] Found {len(plys)} .ply files in {input_dir}')
print(f'[convert] Output formats: ' + ', '.join(fmt for fmt, on in
      [('SOG', WRITE_SOG), ('SPZ', WRITE_SPZ), ('GLB', WRITE_GLB), ('PLY', WRITE_PLY)] if on))
print(f'[convert] Decimate to: {DECIMATE_TO if DECIMATE_TO else "keep all"} | SH bands: {STRIP_SH_BANDS if STRIP_SH_BANDS else "keep all"}')
print(f'[convert] GPU mode: {GPU_MODE}\n')

def _run_st(args, cwd=None):
    """Run splat-transform CLI and return (returncode, stdout, stderr)."""
    return subprocess.run(['splat-transform'] + args,
                             capture_output=True, text=True, cwd=cwd)

# Map format → (ext, build_args, is_lossless)
# Honor ENGINE_PRESET from step2b (cell runs before this one).
# When the preset is OFF, the preset globals are None and we keep
# the existing WRITE_* / STRIP_SH_BANDS / RE_GROUND_ON_IMPORT values
# from this cell's param block (default behavior, no regression).
# When the preset is ON, the globals override these unless they're already
# explicitly True (so user-set True values aren't clobbered by a False preset default).
try:
    _preset_active = ENGINE_PRESET
except NameError:
    _preset_active = False
if _preset_active:
    if _preset_write_sog is not None:
        WRITE_SOG = _preset_write_sog
    if _preset_write_spz is not None:
        WRITE_SPZ = _preset_write_spz
    if _preset_write_glb is not None:
        WRITE_GLB = _preset_write_glb
    if _preset_write_ply is not None:
        WRITE_PLY = _preset_write_ply
    if _preset_sh_bands is not None:
        STRIP_SH_BANDS = _preset_sh_bands
    if _preset_grounding is not None:
        RE_GROUND_ON_IMPORT = _preset_grounding
    print(f'[preset] Applied engine-preset overrides to STEP 3 params.')

formats = []
if WRITE_SOG:
    args = ['-w', '-f', 'sog']
    if DECIMATE_TO > 0:
        args += ['-F', str(DECIMATE_TO)]
    if STRIP_SH_BANDS > 0:
        args += ['-H', str(STRIP_SH_BANDS)]
    if not GPU_MODE:
        args += ['-g', 'cpu']
    formats.append(('sog', '.sog', args, False))
if WRITE_SPZ:
    args = ['-w', '-f', 'spz']
    if DECIMATE_TO > 0:
        args += ['-F', str(DECIMATE_TO)]
    formats.append(('spz', '.spz', args, False))
if WRITE_GLB:
    args = ['-w', '-f', 'glb']
    if DECIMATE_TO > 0:
        args += ['-F', str(DECIMATE_TO)]
    formats.append(('glb', '.glb', args, False))
if WRITE_PLY:
    # PLY passthrough — no decimation or SH strip (would change file semantics)
    formats.append(('ply', '.ply', ['-w', '-f', 'ply'], True))

if not formats:
    raise SystemExit('[convert] All output formats disabled. Enable at least one.')

ok = fail = 0
total_saved_mb = 0.0
t_start = time.time()
results = []
ok = fail = skipped = 0
total_saved_mb = 0.0
t_start = time.time()
results = []
errors_log = []  # (source_path, last_stderr) tuples for the final summary

def _process_one(ply):
    """Process one PLY. Returns a dict with manifest entry + status."""
    import numpy as np
    try:
        orig_size = ply.stat().st_size
    except Exception as e:
        return {'ply': ply, 'ok': False, 'error': f'stat: {e}'}
    # Slug
    if RECURSIVE and ply.parent != input_dir:
        ps = ''.join(c if c.isalnum() else '_' for c in ply.parent.name[:20]).strip('_')
        ss = ''.join(c if c.isalnum() else '_' for c in ply.stem[:40]).strip('_')
        base = f'{ps}_{ss}' if ps else ss
    else:
        base = ply.stem
    base = base[:60]
    # Read source splat count + bbox (best-effort)
    source_splats = None
    source_bbox = None
    try:
        with open(ply, 'rb') as _f:
            _header = _f.read(8192).decode('latin-1', errors='ignore')
        if 'element vertex' in _header:
            for _line in _header.split('\n'):
                if _line.startswith('element vertex'):
                    try:
                        source_splats = int(_line.split()[-1])
                    except ValueError:
                        pass
                    break
    except Exception:
        pass
    # Resume: skip if all enabled outputs exist
    enabled_exts = [ext for fmt_name, ext, _args, _ in formats]
    if RESUME and not FORCE_REDO:
        all_present = True
        for ext in enabled_exts:
            p = output_dir / f'{base}.{fmt_name_for_ext(ext)}'
            if not p.exists() or p.stat().st_size == 0:
                all_present = False
                break
        if all_present and enabled_exts:
            # Load existing sizes into manifest without re-running
            outputs = []
            for fmt_name, ext, _args, _ in formats:
                p = output_dir / f'{base}.{fmt_name}'
                if p.exists():
                    sz = p.stat().st_size
                    outputs.append({
                        'format': fmt_name, 'path': str(p.relative_to(output_dir)),
                        'size_bytes': sz,
                        'compression_ratio': (sz / orig_size * 100) if orig_size else 0,
                    })
            return {'ply': ply, 'base': base, 'ok': True, 'skipped': True,
                    'source_size_bytes': orig_size, 'source_splats': source_splats,
                    'source_bbox': source_bbox, 'outputs': outputs, 'errors': []}
    written = {}
    errors = []
    for fmt_name, ext, base_args, is_lossless in formats:
        out_path = output_dir / f'{base}.{fmt_name}'
        # Skip non-enabled outputs already present (RESUME)
        if RESUME and not FORCE_REDO and out_path.exists() and out_path.stat().st_size > 0:
            sz = out_path.stat().st_size
            written[fmt_name] = (out_path, sz, (sz / orig_size * 100) if orig_size else 0, 0.0, source_splats)
            continue
        args = base_args + [str(ply), str(out_path)]
        t = time.time()
        try:
            r = _run_st(args)
        except subprocess.TimeoutExpired:
            errors.append(f'{fmt_name}: timeout')
            continue
        if r.returncode == 0 and out_path.exists():
            sz = out_path.stat().st_size
            ratio = (sz / orig_size * 100) if orig_size else 0
            written[fmt_name] = (out_path, sz, ratio, time.time() - t, source_splats)
            # Validate SOG outputs
            if VALIDATE_OUTPUTS and ext == '.sog':
                try:
                    import numpy as _np
                    with open(out_path, 'rb') as _vf:
                        _hdr = _vf.read(4096)
                    if b'meta.json' not in _hdr and b'ply' not in _hdr.lower() and len(_hdr) < 64:
                        errors.append(f'{fmt_name}: validation failed (file too small)')
                        written.pop(fmt_name, None)
                except Exception as _ve:
                    errors.append(f'{fmt_name}: validate exception: {_ve}')
        else:
            err = r.stderr.strip().split('\n')[-1] if r.stderr.strip() else r.stdout.strip()
            errors.append(f'{fmt_name}: {err[:200]}')
    # Convert written tuples to manifest-style outputs
    outputs = []
    for fmt_name, (path, size, ratio, dt, splats) in written.items():
        if not path.exists():
            continue
        outputs.append({
            'format': fmt_name,
            'path': str(path.relative_to(output_dir)),
            'size_bytes': size,
            'compression_ratio': round(ratio, 2),
            'elapsed_s': round(dt, 2),
        })
    ok_result = bool(written) and not (errors and len(written) == 0)
    return {'ply': ply, 'base': base, 'ok': ok_result, 'skipped': False,
            'source_size_bytes': orig_size, 'source_splats': source_splats,
            'source_bbox': source_bbox, 'outputs': outputs, 'errors': errors}

# Helper to format the ext->fmt lookup for resume check
def fmt_name_for_ext(ext):
    m = {'sog': 'sog', '.sog': 'sog', 'spz': 'spz', '.spz': 'spz',
         'glb': 'glb', '.glb': 'glb', 'ply': 'ply', '.ply': 'ply'}
    return m.get(ext, ext.lstrip('.'))

# Run sequentially or with K workers
if PARALLEL_WORKERS <= 1:
    _iter = tqdm(plys, desc='convert')
    _results = [_process_one(p) for p in _iter]
else:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    _results = []
    with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as _ex:
        _futures = {_ex.submit(_process_one, p): p for p in plys}
        for _f in tqdm(as_completed(_futures), total=len(_futures), desc=f'convert (x{PARALLEL_WORKERS})'):
            _results.append(_f.result())

# Summarize
for r in _results:
    if r.get('skipped'):
        skipped += 1
    elif r.get('ok'):
        ok += 1
        for out in r['outputs']:
            sz = out['size_bytes']
            orig = r['source_size_bytes']
            if out['format'] != 'ply':
                total_saved_mb += (orig - sz) / 1024 / 1024
    else:
        fail += 1
    results.append(r)
    if r.get('errors'):
        errors_log.append((str(r['ply']), r['errors']))
    # Console report per asset (one line each for serial readability)
    _ply = r['ply']
    _base = r.get('base', _ply.stem)
    if r.get('skipped'):
        print(f'  [skip] {_base} ({len(r["outputs"])} outputs already exist)')
    elif r.get('ok') and r['outputs']:
        _sizes = ', '.join(f"{o['format']}={o['size_bytes']/1024:.0f}KB" for o in r['outputs'])
        _ratio = (sum(o['size_bytes'] for o in r['outputs']) / max(r['source_size_bytes'], 1) * 100)
        print(f'  [ok] {_base}: {len(r["outputs"])} formats ({_sizes}), {_ratio:.0f}% of original')
    else:
        _first_err = (r.get('errors') or ['unknown'])[0][:200]
        print(f'  [fail] {_base}: {_first_err}')
elapsed = time.time() - t_start
print(f'\n[convert] DONE: {ok} OK / {skipped} skipped / {fail} failed in {elapsed:.0f}s')
if total_saved_mb > 0:
    print(f'[convert] Total saved via compression: {total_saved_mb:.0f} MB')
print(f'[convert] Outputs in {output_dir}/')

# Per-file error log (debuggable summary for long runs)
if errors_log:
    print(f"\n[convert] Per-file error log ({len(errors_log)} file(s) had errors):")
    for _src, _errs in errors_log[:20]:
        _short = _src.split("/")[-1]
        print(f"  {_short}: {_errs[0][:120]}")
    if len(errors_log) > 20:
        print(f'  ... and {len(errors_log) - 20} more (re-run with smaller MAX_ITEMS to see)')

# Emit manifest.json (per-source metadata + assetClass inference)
if EMIT_MANIFEST and results:
    _manifest = {
        'version': 1,
        'library_dir': str(input_dir),
        'output_dir': str(output_dir),
        'asset_class_threshold': ASSET_CLASS_THRESHOLD,
        'generated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'summary': {
            'total_sources': len(results),
            'succeeded': ok + skipped,
            'failed': fail,
        },
        'assets': [],
        'errors': [{'source': s, 'errors': e[:3]} for s, e in errors_log],
    }
    for r in results:
        src_splats = r.get('source_splats') or 0
        asset_class = 'environment' if src_splats > ASSET_CLASS_THRESHOLD else 'prop'
        total_out = sum(o['size_bytes'] for o in r.get('outputs', []))
        _entry = {
            'source': str(r['ply']),
            'base': r.get('base', r['ply'].stem),
            'source_size_bytes': r['source_size_bytes'],
            'source_splats': src_splats,
            'assetClass': asset_class,
            'outputs': r.get('outputs', []),
            'outputs_total_bytes': total_out,
            'errors': r.get('errors', []),
            'status': ('skipped' if r.get('skipped') else 'ok' if r.get('ok') else 'failed'),
        }
        if r['source_size_bytes']:
            _entry['overall_compression_ratio'] = round(total_out / r['source_size_bytes'] * 100, 2)
        _manifest['assets'].append(_entry)
    _manifest_path = output_dir / 'manifest.json'
    with open(_manifest_path, 'w') as _mf:
        json.dump(_manifest, _mf, indent=2)
    print(f'[convert] Manifest: {_manifest_path} ({len(_manifest["assets"])} entries)')

# Drive mirror
if DRIVE_MIRROR and ok > 0:
    drive_dest = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/SplatTransform') / output_dir.name
    if drive_dest.parent.exists():
        try:
            drive_dest.mkdir(parents=True, exist_ok=True)
            n = 0
            for f in output_dir.iterdir():
                if f.is_file():
                    shutil.copy2(f, drive_dest / f.name)
                    n += 1
            print(f'[convert] Mirrored {n} files to {drive_dest}')
        except Exception as e:
            print(f'[convert] Drive mirror failed: {e}')
    else:
        print('[convert] Drive not mounted; skipping mirror')

if ok > 0:
    print(f'[convert] Tip: open the Asset_Library_Browser_Colab to browse / tag / preview the converted files.')


In [ ]:
# @title STEP 4 — Decimate + SH-Strip (low-LOD / web preview)
# @markdown Re-process a folder of PLYs with decimation (reduce Gaussian count)
# @markdown and/or SH-stripping (drop higher-frequency color bands). Useful for:
# @markdown   - **Web previews**: 50k-100k Gaussians load in <2 s on consumer GPUs.
# @markdown   - **Mobile**: SH-strip to 0 (DC only) cuts size 50%+ with minor
# @markdown     color loss in close-up renders.
# @markdown   - **LOD chains**: emit high/mid/low versions of each subject.
# @markdown   - **Asset variant generator**: same subject at 4 quality tiers
# @markdown     (0.5x / 0.25x / 0.1x / 0.05x Gaussian count) for a 'game asset pack'.

import os, sys, time, json, shutil, subprocess, pathlib
from tqdm import tqdm

INPUT_DIR = '/content/drive/MyDrive/AEI_3D_Out/TripoSplat'  # @param {type:"string"}
# @markdown *Folder containing TripoSplat .ply files to decimate.*
OUTPUT_DIR = '/content/splat_lite'  # @param {type:"string"}
# @markdown *Output folder. Each input .ply becomes <stem>_LOD<X>.<ext>.*
RECURSIVE = True  # @param {type:"boolean"}
MAX_ITEMS = 0  # @param {type:"integer"}
OUTPUT_FORMAT = 'sog'  # @param ["sog", "spz", "glb", "ply"]
# @markdown *Output format for the decimated versions. SOG is recommended.*
LOD_CHAIN = '131072,65536,32768'  # @param {type:"string"}
# @markdown *Comma-separated Gaussian counts. Each input becomes one output per count. '131072,65536,32768' = 3 LODs (high/mid/low).*'
SH_BANDS = 0  # @param {type:"slider", min:0, max:3, step:1}
# @markdown *0 = keep all SH bands (default). 1-3 = drop higher bands. 3 = DC only (smallest, fastest).*
GPU_MODE = True  # @param {type:"boolean"}
# @markdown *Use GPU for SOG k-means (5-10x faster).*
DRIVE_MIRROR = True  # @param {type:"boolean"}

input_dir = pathlib.Path(INPUT_DIR)
output_dir = pathlib.Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
if not input_dir.exists():
    raise SystemExit(f'[decimate] {input_dir} not found. Run STEP 3 first or change INPUT_DIR.')

# Parse LOD chain
try:
    lod_counts = [int(x.strip()) for x in LOD_CHAIN.split(',') if x.strip()]
    lod_counts = [c for c in lod_counts if c > 0]
except ValueError:
    raise SystemExit(f'[decimate] LOD_CHAIN must be comma-separated integers, got: {LOD_CHAIN!r}')
if not lod_counts:
    raise SystemExit('[decimate] LOD_CHAIN is empty. Provide at least one count.')

# Find PLYs
if RECURSIVE:
    plys = sorted(input_dir.rglob('*.ply'))
else:
    plys = sorted(input_dir.glob('*.ply'))
if not plys:
    raise SystemExit(f'[decimate] No .ply files in {input_dir}.')
if MAX_ITEMS:
    plys = plys[:MAX_ITEMS]
print(f'[decimate] Found {len(plys)} PLYs')
print(f'[decimate] LOD chain: {lod_counts} Gaussians per asset')
print(f'[decimate] Output format: {OUTPUT_FORMAT} | SH bands: {SH_BANDS} | GPU: {GPU_MODE}')
print(f'[decimate] Output: {output_dir}/\n')

def _run_st(args):
    return subprocess.run(['splat-transform'] + args, capture_output=True, text=True)

ok = fail = 0
t_start = time.time()
for ply in tqdm(plys, desc='decimate'):
    try:
        if RECURSIVE and ply.parent != input_dir:
            parent_slug = ''.join(c if c.isalnum() else '_' for c in ply.parent.name[:20]).strip('_')
            stem_slug = ''.join(c if c.isalnum() else '_' for c in ply.stem[:40]).strip('_')
            base = f'{parent_slug}_{stem_slug}' if parent_slug else stem_slug
        else:
            base = ply.stem
        base = base[:60]
        written = []
        for i, count in enumerate(lod_counts):
            out_name = f'{base}_LOD{i}_{count}.{OUTPUT_FORMAT}'
            out_path = output_dir / out_name
            args = ['-w', '-f', OUTPUT_FORMAT, '-F', str(count)]
            if SH_BANDS > 0:
                args += ['-H', str(SH_BANDS)]
            if not GPU_MODE:
                args += ['-g', 'cpu']
            args += [str(ply), str(out_path)]
            t = time.time()
            r = _run_st(args)
            if r.returncode == 0 and out_path.exists():
                size = out_path.stat().st_size
                written.append((i, count, out_path, size, time.time() - t))
            else:
                err = r.stderr.strip().split('\n')[-1] if r.stderr.strip() else r.stdout.strip()
                print(f'  [skip LOD{i}] {err[:200]}')
        if written:
            ok += 1
            lines = [f'  [{ok+fail}/{len(plys)}] {ply.name} -> {len(written)} LODs']
            for i, count, path, size, dt in written:
                lines.append(f'    LOD{i} ({count:,} splats): {size/1024/1024:.1f} MB ({dt:.1f}s)')
            print('\n'.join(lines))
        else:
            fail += 1
            print(f'  [{ok+fail}/{len(plys)}] FAILED: {ply.name}')
    except Exception as e:
        fail += 1
        print(f'  [{ok+fail}/{len(plys)}] EXCEPTION on {ply.name}: {e}')

elapsed = time.time() - t_start
print(f'\n[decimate] DONE: {ok} OK / {fail} failed in {elapsed:.0f}s')
print(f'[decimate] Output: {output_dir}/')

if DRIVE_MIRROR and ok > 0:
    drive_dest = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/SplatTransform') / output_dir.name
    if drive_dest.parent.exists():
        try:
            drive_dest.mkdir(parents=True, exist_ok=True)
            n = 0
            for f in output_dir.iterdir():
                if f.is_file():
                    shutil.copy2(f, drive_dest / f.name)
                    n += 1
            print(f'[decimate] Mirrored {n} files to {drive_dest}')
        except Exception as e:
            print(f'[decimate] Drive mirror failed: {e}')
    else:
        print('[decimate] Drive not mounted; skipping mirror')

if ok > 0:
    print(f'[decimate] Tip: load the highest LOD in PlayCanvas, the middle for editor preview, the lowest for web streaming.')


In [ ]:
# @title STEP 5 — Density Chain (per-tier full-model SOGs, NOT streaming)
# @markdown Generates a per-asset LOD chain by decimating one source PLY into
# @markdown N SOGs of progressively lower Gaussian counts. Each tier is a
# @markdown separate full-model SOG (NOT chunked/spatially-streamed). Best for:
# @markdown   - per-asset LOD chains that an engine swaps in/out based on distance
# @markdown   - distance-based quality switching (hero near = LOD0, far = LOD3)
# @markdown
# @markdown **NOT for streamed progressive loading of large captures.** For that,
# @markdown use STEP 5b (Streamed-SOG Octree) which builds native `lod-meta.json`
# @markdown with spatial tree chunks via splat-transform's `-l` LOD tagging + `lod-meta.json` output.
import os, sys, time, json, shutil, subprocess, pathlib
from tqdm import tqdm

INPUT_DIR = '/content/drive/MyDrive/AEI_3D_Out/TripoSplat'  # @param {type:"string"}
# @markdown *Folder containing TripoSplat .ply files. One LOD chain per file.*
OUTPUT_DIR = '/content/splat_lod'  # @param {type:"string"}
# @markdown *Where to write the LOD chain. One subfolder per asset.*
LOD_COUNTS = '200000,100000,50000,20000'  # @param {type:"string"}
# @markdown *Comma-separated Gaussian counts. 4 LODs = 200k/100k/50k/20k. The first is the highest detail.*
DRIVE_MIRROR = True  # @param {type:"boolean"}

input_dir = pathlib.Path(INPUT_DIR)
output_dir = pathlib.Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
if not input_dir.exists():
    raise SystemExit(f'[lod] {input_dir} not found.')
try:
    lod_counts = [int(x.strip()) for x in LOD_COUNTS.split(',') if x.strip()]
    lod_counts = sorted(set(c for c in lod_counts if c > 0), reverse=True)
except ValueError:
    raise SystemExit(f'[lod] LOD_COUNTS must be comma-separated integers, got: {LOD_COUNTS!r}')
if not lod_counts:
    raise SystemExit('[lod] LOD_COUNTS is empty.')
print(f'[lod] LOD chain: {lod_counts} (descending detail)')

plys = sorted(input_dir.rglob('*.ply'))
if not plys:
    raise SystemExit(f'[lod] No .ply files in {input_dir}.')
print(f'[lod] Found {len(plys)} PLYs\n')

def _run_st(args):
    return subprocess.run(['splat-transform'] + args, capture_output=True, text=True)

ok = fail = 0
t_start = time.time()
for ply in tqdm(plys, desc='lod'):
    try:
        if ply.parent != input_dir:
            parent_slug = ''.join(c if c.isalnum() else '_' for c in ply.parent.name[:20]).strip('_')
            stem_slug = ''.join(c if c.isalnum() else '_' for c in ply.stem[:40]).strip('_')
            base = f'{parent_slug}_{stem_slug}' if parent_slug else stem_slug
        else:
            base = ply.stem
        base = base[:60]
        asset_dir = output_dir / base
        asset_dir.mkdir(parents=True, exist_ok=True)

        # Build each LOD as a separate .sog
        lod_files = []
        for count in lod_counts:
            lod_path = asset_dir / f'{base}_LOD{count}.sog'
            r = _run_st(['-w', '-f', 'sog', '-F', str(count), str(ply), str(lod_path)])
            if r.returncode == 0 and lod_path.exists():
                lod_files.append((count, lod_path))
        if not lod_files:
            fail += 1
            print(f'  [skip] {ply.name}: no LODs generated')
            continue

        # Write lod-meta.json (PlayCanvas viewer's format)
        meta = {
            'name': base,
            'version': 1,
            'lods': [
                {'index': i, 'count': count, 'file': f'{base}_LOD{count}.sog',
                 'size_bytes': path.stat().st_size}
                for i, (count, path) in enumerate(lod_files)
            ],
        }
        (asset_dir / 'lod-meta.json').write_text(json.dumps(meta, indent=2))
        ok += 1
        total_mb = sum(p.stat().st_size for _, p in lod_files) / 1024 / 1024
        print(f'  [ok] {ply.name} -> {base}/ ({"+ ".join(str(c) for c, _ in lod_files)} Gaussians, {total_mb:.1f} MB total)')
    except Exception as e:
        fail += 1
        print(f'  [err] {ply.name}: {e}')

elapsed = time.time() - t_start
print(f'\n[lod] DONE: {ok} OK / {fail} failed in {elapsed:.0f}s')
print(f'[lod] Output: {output_dir}/')

if DRIVE_MIRROR and ok > 0:
    drive_dest = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/SplatTransform') / output_dir.name
    if drive_dest.parent.exists():
        try:
            drive_dest.mkdir(parents=True, exist_ok=True)
            n = 0
            for f in output_dir.rglob('*'):
                if f.is_file():
                    rel = f.relative_to(output_dir)
                    dst = drive_dest / rel
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(f, dst)
                    n += 1
            print(f'[lod] Mirrored {n} files to {drive_dest}')
        except Exception as e:
            print(f'[lod] Drive mirror failed: {e}')
    else:
        print('[lod] Drive not mounted; skipping mirror')


In [ ]:
# @title STEP 5b — Streamed-SOG Octree (native `lod-meta.json`, for environment captures)
# @markdown Builds a **native streamed-SOG octree** via splat-transform's `-l <level>`
# @markdown tagging + `lod-meta.json` output. This is the format the WebGPU/PlayCanvas
# @markdown engine ingests directly for progressive streaming of large captures.
# @markdown
# @markdown **Difference from STEP 5 (density chain):**
# @markdown - STEP 5: N full-model SOGs of decreasing Gaussian counts. The engine
# @markdown   swaps between them based on camera distance. Good for **props**.
# @markdown - STEP 5b: One `lod-meta.json` + spatial chunks of SOGs at decreasing
# @markdown   Gaussian counts. The engine **progressively streams chunks** from low
# @markdown   to high LOD as the camera explores the scene. Good for **environments**.
# @markdown
# @markdown **Workflow (per source PLY):**
# @markdown 1. Decimate the source into N LOD levels via repeated `-F` (kept as PLYs
# @markdown    in a temp dir; tiny because each PLY is just the splat count + XYZ/colors).
# @markdown 2. Convert each decimated PLY to a single-SOG temp file (also small).
# @markdown 3. Bundle them all into a streamed-SOG octree:
# @markdown    `splat-transform -l 0 lod0.sog -l 1 lod1.sog ... <out>/lod-meta.json -C <chunk> -X <extent>`
# @markdown 4. Mirror the result folder to Drive.
# @markdown
# @markdown **Output filename requirement:** the output file MUST be exactly
# @markdown `lod-meta.json` (only the directory path before it can vary). splat-transform
# @markdown rejects any other name for the streamed-SOG format.
INPUT_DIR = '/content/drive/MyDrive/AEI_3D_Out/TripoSplat'  # @param {type:"string"}
# @markdown *Folder containing TripoSplat .ply files. One streamed octree per file.*
OUTPUT_DIR = '/content/splat_streamed'  # @param {type:"string"}
# @markdown *Where to write the streamed-SOG octrees. One subfolder per asset.*
LOD_COUNTS = '200000,100000,50000,20000'  # @param {type:"string"}
# @markdown *Comma-separated Gaussian counts per LOD level. LOD 0 = highest detail.
# @markdown 4 levels = 200k/100k/50k/20k. For 2M+ source: 1000000,500000,200000,50000.*
LOD_CHUNK_COUNT = 512  # @param {type:"integer"}
# @markdown *Approximate Gaussians per chunk (in K). Default 512 = ~512k splats per chunk.
# @markdown Lower = smaller chunks + more chunks = finer streaming granularity.*
LOD_CHUNK_EXTENT = 16  # @param {type:"integer"}
# @markdown *Approximate chunk size in world units (meters). Default 16m. Increase for
# @markdown very large scenes to avoid generating a huge number of small chunks.*
STRIP_SH_BANDS = 3  # @param {type:"slider", min:0, max:3, step:1}
# @markdown *SH bands to strip from ALL LOD tiers (saves bytes). 3 = DC-only.
# @markdown Engine renders DC + degree-1; if your shader does that, use 2.*
APPLY_GROUNDING = False  # @param {type:"boolean"}
# @markdown *Pre-ground the source PLY before LOD generation. RECOMMENDED for
# @markdown environment captures (they load in world coords). LEAVE OFF for props
# @markdown (the engine auto-orients them; pre-grounding double-applies).*
DRIVE_MIRROR = True  # @param {type:"boolean"}
# @markdown *Mirror OUTPUT_DIR to /content/drive/MyDrive/AEI_3D_Out/SplatTransform/Streamed at end.*
FORCE_REDO = False  # @param {type:"boolean"}
# @markdown *Force regeneration even if `lod-meta.json` already exists in the output dir.*

import os, sys, time, json, shutil, subprocess, tempfile, pathlib
from tqdm import tqdm

_step5b_input_dir = pathlib.Path(INPUT_DIR)
_step5b_output_dir = pathlib.Path(OUTPUT_DIR)
_step5b_output_dir.mkdir(parents=True, exist_ok=True)
if not _step5b_input_dir.exists():
    raise SystemExit(f'[5b] {_step5b_input_dir} not found. Run TripoSPlat STEP 7 first.')
RECURSIVE_GUARD = True
_step5b_plys = sorted(_step5b_input_dir.rglob('*.ply')) if RECURSIVE_GUARD else sorted(_step5b_input_dir.glob('*.ply'))
if not _step5b_plys:
    raise SystemExit(f'[5b] No .ply files in {_step5b_input_dir}. Run TripoSPlat first.')
try:
    _step5b_lod_counts = [int(x.strip()) for x in LOD_COUNTS.split(',') if x.strip()]
except ValueError:
    raise SystemExit(f'[5b] LOD_COUNTS must be comma-separated integers, got: {LOD_COUNTS!r}')
if len(_step5b_lod_counts) < 2:
    raise SystemExit(f'[5b] LOD_COUNTS must have at least 2 levels, got {len(_step5b_lod_counts)}.')

# RECURSIVE_GUARD isn't a param — default to True since TripoSplat outputs are nested

def _step5b_run_st(args, timeout=600):
    return subprocess.run(['splat-transform'] + args, capture_output=True, text=True, timeout=timeout)

def _step5b_apply_grounding(ply_in, ply_out):
    """Reuse the same grounding helper TripoSPlat STEP 8 uses."""
    try:
        # Lazy import — may not be available in fresh Colab runtime
        import numpy as np
        import open3d as o3d
    except ImportError as e:
        print(f'[5b] WARN: grounding deps missing ({e}); skipping grounding for {ply_in.name}')
        shutil.copy2(str(ply_in), str(ply_out))
        return False
    try:
        pcd = o3d.io.read_point_cloud(str(ply_in))
        pts = np.asarray(pcd.points)
        if len(pts) < 10:
            shutil.copy2(str(ply_in), str(ply_out))
            return False
        # Translate to origin in XZ
        cx, cz = (pts[:, 0].min() + pts[:, 0].max()) / 2, (pts[:, 2].min() + pts[:, 2].max()) / 2
        pts[:, 0] -= cx
        pts[:, 2] -= cz
        # Translate to Y=0
        ymin = pts[:, 1].min()
        pts[:, 1] -= ymin
        pcd.points = o3d.utility.Vector3dVector(pts)
        o3d.io.write_point_cloud(str(ply_out), pcd)
        return True
    except Exception as e:
        print(f'[5b] WARN: grounding failed for {ply_in.name}: {e}; copying unmodified.')
        shutil.copy2(str(ply_in), str(ply_out))
        return False

ok = fail = skipped = 0
total_t = time.time()
for ply in tqdm(_step5b_plys, desc='streamed-octree'):
    try:
        if RECURSIVE_GUARD and ply.parent != _step5b_input_dir:
            ps = ''.join(c if c.isalnum() else '_' for c in ply.parent.name[:20]).strip('_')
            ss = ''.join(c if c.isalnum() else '_' for c in ply.stem[:40]).strip('_')
            base = f'{ps}_{ss}' if ps else ss
        else:
            base = ply.stem
        base = base[:60]
        asset_dir = _step5b_output_dir / base
        lod_meta_path = asset_dir / 'lod-meta.json'
        # Resume/skip if outputs already exist
        if not FORCE_REDO and lod_meta_path.exists() and lod_meta_path.stat().st_size > 0:
            skipped += 1
            print(f'  [skip] {base}: lod-meta.json already exists (set FORCE_REDO=True to regenerate).')
            continue
        asset_dir.mkdir(parents=True, exist_ok=True)
        # Temp dir for decimated PLYs (cleaned up after)
        with tempfile.TemporaryDirectory(prefix='streamed_octree_') as tmpdir:
            tmp = pathlib.Path(tmpdir)
            # Optional grounding first
            if APPLY_GROUNDING:
                grounded = tmp / f'{base}_grounded.ply'
                if _step5b_apply_grounding(ply, grounded):
                    source_for_decimation = grounded
                else:
                    source_for_decimation = ply
            else:
                source_for_decimation = ply
            # Decimate to each LOD level (output is PLY, tiny because no SH)
            lod_plys = []
            for level, count in enumerate(_step5b_lod_counts):
                lod_ply = tmp / f'{base}_lod{level}.ply'
                args = ['-w', '-f', 'ply', '-F', str(count)]
                if STRIP_SH_BANDS > 0:
                    args += ['-H', str(STRIP_SH_BANDS)]
                args += [str(source_for_decimation), str(lod_ply)]
                r = _step5b_run_st(args, timeout=300)
                if r.returncode == 0 and lod_ply.exists():
                    lod_plys.append((level, lod_ply))
                else:
                    err = r.stderr.strip().split(chr(10))[-1] if r.stderr.strip() else 'no stderr'
                    print(f'  [warn] {base} LOD{level} ({count} G) decimation failed: {err[:100]}')
            if not lod_plys:
                fail += 1
                print(f'  [fail] {base}: no LOD tiers generated.')
                continue
            # Convert each decimated PLY to a single-SOG temp file
            lod_sogs = []
            for level, lod_ply in lod_plys:
                lod_sog = tmp / f'{base}_lod{level}.sog'
                args = ['-w', '-f', 'sog']
                if STRIP_SH_BANDS > 0:
                    args += ['-H', str(STRIP_SH_BANDS)]
                args += [str(lod_ply), str(lod_sog)]
                r = _step5b_run_st(args, timeout=600)
                if r.returncode == 0 and lod_sog.exists():
                    lod_sogs.append((level, lod_sog))
                else:
                    err = r.stderr.strip().split(chr(10))[-1] if r.stderr.strip() else 'no stderr'
                    print(f'  [warn] {base} LOD{level} SOG conversion failed: {err[:100]}')
            if not lod_sogs:
                fail += 1
                print(f'  [fail] {base}: no SOG conversions succeeded.')
                continue
            # Bundle into streamed-SOG octree via -l tagging
            args = []
            for level, lod_sog in lod_sogs:
                args += [str(lod_sog), '-l', str(level)]
            args += [str(lod_meta_path), '-C', str(LOD_CHUNK_COUNT), '-X', str(LOD_CHUNK_EXTENT)]
            r = _step5b_run_st(args, timeout=900)
            if r.returncode != 0:
                err = r.stderr.strip().split(chr(10))[-1] if r.stderr.strip() else 'no stderr'
                fail += 1
                print(f'  [fail] {base}: octree bundling failed: {err[:200]}')
                continue
            if not lod_meta_path.exists():
                fail += 1
                print(f'  [fail] {base}: lod-meta.json not written.')
                continue
            # Compute total bytes for the report
            total_bytes = sum(p.stat().st_size for _, p in lod_sogs) + lod_meta_path.stat().st_size
            counts_str = '+'.join(str(_step5b_lod_counts[lvl]) for lvl, _ in lod_sogs)
            ok += 1
            print(f'  [ok] {base}: {len(lod_sogs)} LODs ({counts_str} G), {total_bytes/1024/1024:.1f} MB total -> {lod_meta_path}')
    except Exception as e:
        fail += 1
        print(f'  [err] {ply.name}: {e}')
elapsed = time.time() - total_t
print(f'\n[5b] DONE: {ok} OK / {skipped} skipped / {fail} failed in {elapsed:.0f}s')
if ok > 0:
    print(f'[5b] Output: {_step5b_output_dir}/')
    print('[5b] Each asset folder contains lod-meta.json + per-LOD .sog chunks. Upload to engine as-is.')
    if DRIVE_MIRROR:
        drive_dest = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/SplatTransform/Streamed') / _step5b_output_dir.name
        if drive_dest.parent.exists():
            try:
                if drive_dest.exists():
                    shutil.rmtree(drive_dest)
                shutil.copytree(_step5b_output_dir, drive_dest)
                n_files = sum(1 for _ in drive_dest.rglob('*') if _.is_file())
                print(f'[5b] Mirrored {n_files} files to {drive_dest}')
            except Exception as e:
                print(f'[5b] Drive mirror failed: {e}')
        else:
            print('[5b] Drive not mounted; skipping mirror')


In [ ]:
# @title STEP 6 — Collider Generation: Voxel / Convex Hull / Alpha Shape
# @markdown Three non-game-asset outputs from splat-transform + trimesh:
# @markdown   - `.collision.glb` — marching-cubes mesh from voxelizing the
# @markdown     splat cloud. Use for **runtime physics** (collision detection).
# @markdown     NOT a textured visual mesh. Resolution tunable 32-512.
# @markdown   - `.hull.glb` — convex hull wrapping the splat cloud. Tiny
# @markdown     (~100-1000 vertices), perfect for simple props, spheres,
# @markdown     anything blob-shaped. **Best for low-LOD background assets.**
# @markdown   - `.concave.glb` — alpha-shape (concave hull) wrapping the
# @markdown     splat cloud. Follows surface concavities. Best for organic
# @markdown     meshes (characters, weapons, anything lumpy). Configurable
# @markdown     detail via the `ALPHA_FACTOR` slider.
# @markdown
# @markdown **Which one should I use?**
# @markdown - Hero assets (5-10): use full GauStudio recon as the collider
# @markdown   (NOT this step). See GauStudio_Colab.
# @markdown - Standard assets (30-50): use voxel collision at 128^3 (~1-3 MB).
# @markdown - Background / decorative (140+): use convex hull (~10-50 KB).
# @markdown - Concave shapes (characters with limbs, weapons with grips):
# @markdown   use alpha shape with ALPHA_FACTOR=2-5.
# @markdown
# @markdown **Engines with built-in collision baking:** if your engine (Unreal,
# @markdown Unity DOTS physics, custom Havok/PhysX setup, etc.) bakes its own collision
# @markdown meshes at scene-import time, you can SKIP this step entirely — the
# @markdown collision `.glb`s won't be needed. Just leave all 3 GENERATE_* toggles False.
# @markdown
# @markdown **Three.js usage:**
# @markdown ```js
# @markdown const collider = await new GLTFLoader().loadAsync('hero.collision.glb');
# @markdown collider.scene.visible = false;  // invisible, used only for raycasting
# @markdown raycaster.intersectObjects(collider.scene.children, true);
# @markdown ```

import os, sys, time, json, shutil, subprocess, pathlib
import numpy as np
from tqdm import tqdm
import trimesh
def _ground_splat(ply_in_path, ply_out_path, *, axis_align=True, center_xz=True,                    subsample_for_pca=20000):
    """Ground a 3DGS PLY: translate so bottom sits on Y=0, center XZ, axis-align.
    Returns transform dict for meta.json."""
    import open3d as o3d
    pcd = o3d.io.read_point_cloud(str(ply_in_path))
    pts = np.asarray(pcd.points)
    if len(pts) < 10:
        shutil.copy2(str(ply_in_path), str(ply_out_path))
        return {"applied": False, "reason": "too few points"}
    n = len(pts)
    if n > subsample_for_pca:
        idx = np.random.choice(n, subsample_for_pca, replace=False)
        pts_pca = pts[idx]
    else:
        pts_pca = pts
    if center_xz:
        cx = float(np.median(pts[:, 0]))
        cz = float(np.median(pts[:, 2]))
    else:
        cx = cz = 0.0
    rot_y = 0.0
    if axis_align:
        pts_xz = pts_pca[:, [0, 2]] - np.array([cx, cz])
        cov = np.cov(pts_xz.T)
        if cov.shape == (2, 2):
            eigvals, eigvecs = np.linalg.eigh(cov)
            principal = eigvecs[:, np.argmax(eigvals)]
            angle = float(np.arctan2(principal[1], principal[0]))
            rot_y = round(angle / (np.pi / 2)) * (np.pi / 2)
    cos_r, sin_r = float(np.cos(-rot_y)), float(np.sin(-rot_y))
    R = np.array([[cos_r, 0, -sin_r],
                  [0,     1, 0],
                  [sin_r, 0, cos_r]])
    pts_centered = pts - np.array([cx, 0.0, cz])
    pts_rot = pts_centered @ R.T
    min_y_after = float(np.min(pts_rot[:, 1]))
    final_pts = pts_rot - np.array([0.0, min_y_after, 0.0])
    out_pcd = o3d.geometry.PointCloud()
    out_pcd.points = o3d.utility.Vector3dVector(final_pts)
    if pcd.has_colors():
        out_pcd.colors = pcd.colors
    o3d.io.write_point_cloud(str(ply_out_path), out_pcd,
                              write_ascii=False, compressed=False)
    bbox_min = final_pts.min(axis=0).tolist()
    bbox_max = final_pts.max(axis=0).tolist()
    height = bbox_max[1] - bbox_min[1]
    width = bbox_max[0] - bbox_min[0]
    depth = bbox_max[2] - bbox_min[2]
    if height < 0.5:
        size_class = "small_prop"
    elif height < 2.0:
        size_class = "prop"
    elif height < 10.0:
        size_class = "tree_or_vehicle"
    else:
        size_class = "building"
    return {
        "applied": True,
        "translation": {"x": cx, "y": float(min_y_after), "z": cz},
        "rotation_y_radians": rot_y,
        "rotation_y_degrees": float(np.degrees(rot_y)),
        "center_xz": bool(center_xz), "axis_align": bool(axis_align),
        "bounds_after": {"min": bbox_min, "max": bbox_max},
        "size_units": {"height": height, "width": width, "depth": depth},
        "size_class": size_class,
    }

def _size_class_for_height(height):
    if height < 0.5: return "small_prop"
    if height < 2.0: return "prop"
    if height < 10.0: return "tree_or_vehicle"
    return "building"


INPUT_DIR = '/content/drive/MyDrive/AEI_3D_Out/TripoSplat'  # @param {type:"string"}
OUTPUT_DIR = '/content/splat_colliders'  # @param {type:"string"}

# @markdown **Voxel collision mesh** (marching-cubes, GPU-only):
GENERATE_VOXEL = True  # @param {type:"boolean"}
# @markdown *Generate `.collision.glb` from the voxel grid. Best for runtime physics, NOT visuals.*
VOXEL_RESOLUTION = 128  # @param {type:"slider", min:32, max:512, step:32}
# @markdown *Voxel grid resolution. 128 = fast, 256 = detailed, 512 = slow but accurate.*

# @markdown **Convex hull** (trimesh, CPU):
GENERATE_HULL = True  # @param {type:"boolean"}
# @markdown *Generate `.hull.glb` as a convex hull wrapping the splat cloud. Tiny (~10-50 KB).*
HULL_MAX_POINTS = 50000  # @param {type:"integer"}
# @markdown *Subsample splats to this count for hull computation. Lower = faster, less accurate.*
RE_GROUND_ON_IMPORT = False  # @param {type:"boolean"}
# @markdown *Re-ground input PLY before collider generation. Use for PLYs from
# @markdown non-TripoSplat sources (GauStudio, hand-made, etc.) that need grounding.*
RE_GROUND_AXIS_ALIGN = True  # @param {type:"boolean"}
# @markdown *PCA axis-align (snap to 90 deg) when re-grounding.*
RE_GROUND_CENTER_XZ = True  # @param {type:"boolean"}
# @markdown *Center XZ when re-grounding.*

# @markdown **Concave hull (alpha shape)** (trimesh, CPU):
GENERATE_CONCAVE = False  # @param {type:"boolean"}
# @markdown *Generate `.concave.glb` via alpha-shape (concave hull). Follows surface concavities.*
ALPHA_FACTOR = 2.0  # @param {type:"slider", min:0.5, max:10.0, step:0.1}
# @markdown *Alpha factor. 0.5 = tight wrap (may miss features), 2.0 = balanced,
# @markdown 5.0+ = close to convex hull. 2-5 is the sweet spot for most splats.*

DRIVE_MIRROR = True  # @param {type:"boolean"}

input_dir = pathlib.Path(INPUT_DIR)
output_dir = pathlib.Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
if not input_dir.exists():
    raise SystemExit(f'[collider] {input_dir} not found.')
plys = sorted(input_dir.rglob('*.ply'))
if not plys:
    raise SystemExit(f'[collider] No .ply files in {input_dir}.')
print(f'[collider] Found {len(plys)} PLYs')
print(f'[collider] Voxel={GENERATE_VOXEL}@{VOXEL_RESOLUTION}^3 | Hull={GENERATE_HULL} | Concave={GENERATE_CONCAVE}@alpha={ALPHA_FACTOR}')

def _run_st(args):
    return subprocess.run(['splat-transform'] + args, capture_output=True, text=True)

def _read_ply_xyz(ply_path):
    """Read just the xyz positions from a 3DGS PLY (skip SH/rotation attrs)."""
    import open3d as o3d
    pcd = o3d.io.read_point_cloud(str(ply_path))
    return np.asarray(pcd.points)

ok = fail = 0
t_start = time.time()
for ply in tqdm(plys, desc='collider'):
    try:
        if ply.parent != input_dir:
            parent_slug = ''.join(c if c.isalnum() else '_' for c in ply.parent.name[:20]).strip('_')
            stem_slug = ''.join(c if c.isalnum() else '_' for c in ply.stem[:40]).strip('_')
            base = f'{parent_slug}_{stem_slug}' if parent_slug else stem_slug
        else:
            base = ply.stem
        base = base[:60]
        written = []

        if GENERATE_VOXEL:
            col_path = output_dir / f'{base}_collision.glb'
            # -K flag: marching-cubes mesh from the voxel grid (GPU only).
            # Optional: re-ground input PLY before generating colliders
            source_ply = ply
            if RE_GROUND_ON_IMPORT:
                grounded_ply = output_dir / f"{base}_grounded.ply"
                try:
                    _ground_splat(ply, grounded_ply,
                                    axis_align=RE_GROUND_AXIS_ALIGN,
                                    center_xz=RE_GROUND_CENTER_XZ)
                    source_ply = grounded_ply
                except Exception as e:
                    print(f"  [grounding fail] {base}: {e}")
            r = _run_st(['-w', '-K', str(VOXEL_RESOLUTION), str(source_ply), str(col_path)])
            if r.returncode == 0 and col_path.exists():
                written.append(('collision.glb', col_path))
            else:
                err = r.stderr.strip().split(chr(10))[-1] if r.stderr.strip() else ''
                print(f'  [skip voxel] {base}: {err[:120]}')

        if GENERATE_HULL or GENERATE_CONCAVE:
            try:
                xyz = _read_ply_xyz(source_ply)
                n_pts = len(xyz)
                if n_pts > HULL_MAX_POINTS:
                    idx = np.random.choice(n_pts, HULL_MAX_POINTS, replace=False)
                    xyz_sub = xyz[idx]
                else:
                    xyz_sub = xyz
                pcd = trimesh.PointCloud(xyz_sub)

                if GENERATE_HULL:
                    try:
                        hull = pcd.convex_hull
                        hull_path = output_dir / f'{base}_hull.glb'
                        hull.export(str(hull_path))
                        written.append(('hull.glb', hull_path))
                    except Exception as e:
                        print(f'  [skip hull] {base}: {e}')

                if GENERATE_CONCAVE:
                    try:
                        # trimesh.alpha_shape returns a Trimesh if radii is given,
                        # else a list. Use a fixed radius based on ALPHA_FACTOR.
                        # Heuristic: r = ALPHA_FACTOR * median_nn_distance
                        from scipy.spatial import cKDTree
                        tree = cKDTree(xyz_sub)
                        nn_dists, _ = tree.query(xyz_sub, k=2)
                        median_nn = float(np.median(nn_dists[:, 1]))
                        radius = ALPHA_FACTOR * median_nn
                        concave = trimesh.creation.alpha_shape(xyz_sub, radius)
                        if isinstance(concave, list):
                            concave = trimesh.util.concatenate(concave)
                        if len(concave.faces) > 0:
                            cpath = output_dir / f'{base}_concave.glb'
                            concave.export(str(cpath))
                            written.append(('concave.glb', cpath))
                        else:
                            print(f'  [skip concave] {base}: alpha_shape returned 0 faces (try lower ALPHA_FACTOR)')
                    except ImportError:
                        print(f'  [skip concave] scipy not available')
                    except Exception as e:
                        print(f'  [skip concave] {base}: {e}')
            except Exception as e:
                print(f'  [skip python collider] {base}: {e}')

        if written:
            ok += 1
            desc = ', '.join(f'{n} ({p.stat().st_size/1024:.0f} KB)' for n, p in written)
            print(f'  [ok] {ply.name} -> {desc}')
        else:
            fail += 1
            print(f'  [skip] {ply.name}: nothing generated')
    except Exception as e:
        fail += 1
        print(f'  [err] {ply.name}: {e}')

elapsed = time.time() - t_start
print(f'\n[collider] DONE: {ok} OK / {fail} failed in {elapsed:.0f}s -> {output_dir}')

if DRIVE_MIRROR and ok > 0:
    drive_dest = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/SplatTransform') / output_dir.name
    if drive_dest.parent.exists():
        try:
            drive_dest.mkdir(parents=True, exist_ok=True)
            n = 0
            for f in output_dir.iterdir():
                if f.is_file():
                    shutil.copy2(f, drive_dest / f.name)
                    n += 1
            print(f'[collider] Mirrored {n} files to {drive_dest}')
        except Exception as e:
            print(f'[collider] Drive mirror failed: {e}')


In [ ]:
# @title STEP 7 — Summary + Final Drive Mirror
# @markdown Lists all outputs produced across STEP 3-6 and mirrors to Drive.

import os, shutil, pathlib
from datetime import datetime

DRIVE_BASE = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/SplatTransform')
LOCAL_OUTPUTS = ['/content/splat_out', '/content/splat_lite', '/content/splat_lod',
                 '/content/splat_voxel']

if not DRIVE_BASE.parent.exists():
    print('[summary] Drive not mounted.')
else:
    print(f'[summary] Drive target: {DRIVE_BASE}')
    print(f'[summary] Local outputs found:')
    grand_total_files = 0
    grand_total_mb = 0.0
    for local in LOCAL_OUTPUTS:
        local_p = pathlib.Path(local)
        if not local_p.exists():
            continue
        files = list(local_p.rglob('*'))
        files = [f for f in files if f.is_file()]
        if not files:
            continue
        total_mb = sum(f.stat().st_size for f in files) / 1024 / 1024
        grand_total_files += len(files)
        grand_total_mb += total_mb
        # Find newest + oldest files
        newest = max(files, key=lambda f: f.stat().st_mtime)
        print(f'  {local}/: {len(files)} files, {total_mb:.1f} MB (newest: {newest.name})')
    print(f'\n[summary] TOTAL: {grand_total_files} files, {grand_total_mb:.1f} MB')
    print(f'[summary] ts: {datetime.now().isoformat()}')

    if grand_total_files > 0:
        # Final mirror: all local outputs go to /SplatTransform/<name>/
        DRIVE_BASE.mkdir(parents=True, exist_ok=True)
        for local in LOCAL_OUTPUTS:
            local_p = pathlib.Path(local)
            if not local_p.exists() or not any(local_p.iterdir()):
                continue
            dest = DRIVE_BASE / local_p.name
            try:
                if dest.exists():
                    shutil.rmtree(dest)
                shutil.copytree(local_p, dest)
                print(f'[summary] Mirrored {local_p.name}/ to {dest}')
            except Exception as e:
                print(f'[summary] Failed to mirror {local_p.name}: {e}')
        # Drop a README in the Drive root summarizing what's where
        readme = DRIVE_BASE / 'README.md'
        readme.write_text(f'''# SplatTransform Exports

Generated {datetime.now().isoformat()} from the TripoSplat + splat-transform pipeline.

## Folders

- `splat_out/` — STEP 3 batch convert: PLY → SOG + SPZ + GLB+KHR_gaussian_splatting + PLY
- `splat_lite/` — STEP 4 decimated + SH-stripped versions for web preview / mobile
- `splat_lod/` — STEP 5 LOD chains (streamed SOG for PlayCanvas progressive loading)
- `splat_voxel/` — STEP 6 voxel collision meshes (for runtime physics, NOT visuals)

## Format reference

- **`.sog`** — PlayCanvas native, ~10x compression, GPU SH k-means
- **`.spz`** — Niantic, smallest cross-platform, mobile AR
- **`.glb` + `KHR_gaussian_splatting`** — glTF 2.0 standard extension. Three.js / PlayCanvas / Babylon.js. Unity / Unreal glTF importer support coming.
- **`.ply`** — lossless, universal fallback (Antimatter15, gsplat.tech, LumaAI)
- **`.collision.glb`** — marching-cubes from voxel grid, for runtime physics

## For textured visual mesh

`splat-transform` does NOT generate textured visual meshes. For that:
- Single image: [Pixal3D_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Pixal3D_Colab.ipynb) (PBR-textured GLB, best quality)
- Hero assets from 3DGS: [SuGaR_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SuGaR_Colab.ipynb) (sharp, 2-3 hrs) or [GauStudio_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/GauStudio_Colab.ipynb) (smooth, ~10 min)
''')
        print(f'[summary] Wrote {readme}')

print(f'\n[summary] Done. Open {DRIVE_BASE}/ in your file browser to see all outputs.')
print(f'[summary] Use Asset_Library_Browser_Colab to browse / tag / preview.')


In [ ]:
# @title STEP 8 — Keep Alive (anti-disconnect)
# @markdown Standard pattern. splat-transform batches are usually <10 min, but
# @markdown if you have 200+ assets or run multiple decimation passes, this
# @markdown prevents Colab from idling you out.

import IPython
from google.colab import output
KEEP_ALIVE_JS = """
function KeepAlive() {
    setInterval(() => {
        console.log("keep-alive:", new Date().toISOString());
        google.colab.kernel.proxyProxy(0).then(() => {}).catch(() => {});
    }, 30 * 60 * 1000);
}
KeepAlive();
"""
display(IPython.display.Javascript(KEEP_ALIVE_JS))
print('[KeepAlive] Active — pings every 30 min.')


In [ ]:
# @title STEP 9 — Help / Format Reference / Engine Compatibility / Known Issues
# @markdown Reference material. Read this if anything in the pipeline went wrong.

help_md = '''
# SplatTransform — Help, Format Reference, and Engine Compatibility

## What this notebook is for

Takes a folder of [TripoSplat_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/TripoSplat_Colab.ipynb) `.ply` outputs (raw 3D Gaussian Splats, ~150-250 MB each) and converts them to game-engine-friendly formats with ~10x compression. This is the **missing piece** for shipping 3DGS assets to a game engine without breaking your Drive storage budget.

## Format reference

| Format | Engine/consumer | Size vs PLY | Use for |
|--------|-----------------|-------------|---------|
**`.sog`** | PlayCanvas engine, PlayCanvas viewer | ~10% (90% smaller) | Production PlayCanvas scenes |
**`.spz`** | Niantic Scaniverse, mobile AR | ~5% | Mobile AR, smallest files |
**`.glb` + `KHR_gaussian_splatting`** | glTF 2.0 standard (Three.js, PlayCanvas, Babylon.js) | ~20% | Future-proof, glTF-native |
**`.ply`** | Antimatter15, gsplat.tech, LumaAI, Unity community splat renderers | 100% (lossless) | Universal fallback, lossless archive |
**`.collision.glb`** | Any glTF 2.0 importer (runtime physics) | n/a | Collision detection (NOT visual) |
**`.webp`** (turntable) | n/a | n/a | Portfolio / Asset_Library_Browser thumbnails |
**`lod-meta.json` + multiple `.sog`** | PlayCanvas viewer (progressive loader) | depends on LODs | Streaming LODs |

## Engine compatibility (Jan 2026)

- **PlayCanvas Engine** ✅ first-class. SOG is native. glTF `KHR_gaussian_splatting` is supported.
- **Three.js** ✅ via `gsplat.js` / `@mkkellogg/gaussian-splats-3d` community plugins. SPZ, KSPLAT, PLY all work.
- **Babylon.js** ✅ has a `GaussianSplatting` component. PLY, SPZ, GLB+KHR_gaussian_splatting.
- **Unity** ⚠️ community splat renderers read PLY/SPLAT/SPZ. The `KHR_gaussian_splatting` glTF extension is brand-new (2024-25) — check your Unity version's glTF importer. If it doesn't recognize the extension yet, ship PLY + SPZ alongside.
- **Unreal Engine** ⚠️ same caveat as Unity. Test with your UE version's glTF importer. SOG is PlayCanvas-specific (won't load in UE).
- **Godot** ⚠️ Godot 4.3+ has experimental splat support via plugins. Test PLY first; GLB+KHR_gaussian_splatting is the better long-term bet.
- **Web (custom viewer)** ✅ drop in [gsplat.tech](https://gsplat.tech/) or [playcanvas.com/viewer](https://playcanvas.com/viewer/).

## Known issues

1. **`splat-transform: command not found` after install**: the global `node_modules` PATH wasn't set. Try `!npx splat-transform ...` instead, or `!npm install -g @playcanvas/splat-transform && !hash -r`.
2. **`EACCES: permission denied` writing to /usr/local/bin**: prefix the install with `sudo` if running outside Colab, or install locally with `!npm install @playcanvas/splat-transform` (without `-g`).
3. **SOG output is huge despite the 10x claim**: check that SH bands weren't stripped to 0 and that the input PLY had SH degree 3 (TripoSplat does). SH degree 0 inputs (no color detail) won't compress much.
4. **GLB with `KHR_gaussian_splatting` won't open in Blender 4.x**: the extension is very new. Blender 4.4+ may recognize it. For Blender, use PLY + external `gsplat` importer.
5. **Voxel collision mesh is huge (>50 MB)**: lower `VOXEL_RESOLUTION` from 128 to 64 or 32. Or filter the input PLY to remove floating-point outliers first (`-F 200000` decimate to 200k splats).
6. **Turntable preview is all black**: requires WebGPU. Colab instances have it, but check `nvidia-smi` for Vulkan.
7. **Drive mirror fails silently**: check that `DRIVE_BASE.parent` (i.e. `/content/drive/MyDrive`) exists. If not, mount Drive in the Colab file browser first.

## Engine Preset (Step 2b)

If you're feeding outputs into a **WebGPU Gaussian-splat game engine** (loads bundled `.sog` SOG v2,
auto-orients assets at load time, renders only DC + degree-1 SH), enable `ENGINE_PRESET = True`
and pick `ASSET_CLASS = "prop"` or `"environment"`. See `step2a-engine-preset-doc` markdown for the full
prop-vs-environment distinction (especially the **don't pre-ground props** caveat).

When `ENGINE_PRESET = False` (default), every existing param behaves exactly as before. No regression for current users.

## STEP 5 vs STEP 5b

- **STEP 5 (density chain):** N full-model SOGs of decreasing Gaussian counts. Engine swaps in/out based on
  camera distance. Best for **object props**. NOT for progressive streaming.
- **STEP 5b (streamed-SOG octree):** one `lod-meta.json` + spatial SOG chunks. Engine progressively streams
  chunks from low to high LOD. Best for **environment captures**. Built via splat-transform's
  native `-l <level>` tagging + `lod-meta.json` output.

## STEP 3 batch hardening

- `PARALLEL_WORKERS` (1-8, default 2): runs K conversions concurrently via `ThreadPoolExecutor`.
- `RESUME` (default True): skips sources whose outputs already exist (survives Colab disconnects).
- `VALIDATE_OUTPUTS` (default True): re-opens each SOG to confirm it parses.
- `EMIT_MANIFEST` (default True): writes `<output_dir>/manifest.json` with per-source metadata +
  `assetClass` (auto-inferred from splat count vs `ASSET_CLASS_THRESHOLD`). Lets the engine's
  upload step automate asset records + LOD variants.

## When to use splat-transform vs other 3D tools in this suite

| Need | Use |
|------|-----|
Single image → 3DGS (raw) | [TripoSplat_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/TripoSplat_Colab.ipynb) |
3DGS → game-ready formats (SOG/SPZ/GLB) | **THIS notebook** |
Single image → textured PBR GLB mesh | [Pixal3D_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Pixal3D_Colab.ipynb) |
3DGS → high-quality research mesh | [SuGaR_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SuGaR_Colab.ipynb) / [GauStudio_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/GauStudio_Colab.ipynb) |
Browse 200+ assets | [Asset_Library_Browser_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Asset_Library_Browser_Colab.ipynb) |
Post-process Pixal3D GLBs | [Pixal3D_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Pixal3D_Colab.ipynb) Step 8 |

## Recommended 200+ library workflow

```
200+ images
  ↓
TripoSplat_Colab Step 7 batch → 200x .ply (~150 MB each = 30 GB total)
  ↓
SplatTransform_Colab Step 3 batch → 200x .sog (~15 MB each = 3 GB total)
  ↓ [saves 27 GB of Drive space]
  ↓
SplatTransform_Colab Step 4 → 200x decimated web previews (~5 MB each)
  ↓
SplatTransform_Colab Step 5 → 200x LOD chains for PlayCanvas streaming
  ↓
Asset_Library_Browser → browse / tag / preview the 600+ files
  ↓
Ship .sog to PlayCanvas / .glb+KHR_gaussian_splatting to Three.js+Three.js / .ply to anyone else
```

## Citation

PlayCanvas `splat-transform`. [GitHub](https://github.com/playcanvas/splat-transform). MIT License (Copyright 2011–2026 PlayCanvas Ltd.).
'''
print(help_md)
